# 04 — Discussion figures and tables

Produces the Discussion-section figures:

| Output | Generator |
|---|---|
| Fig 10 — Robustness of inter-site contrast | `make_results_figures.fig_robustness` |
| Fig 11 — alpha-sweep (Martinez per-site density retrieval) | `make_alpha_sweep_figure.main` |
| Table 4 — Pooled-fit model comparison | inlined below |

Wall time: 5-10 min.

In [ ]:
import sys, json, pathlib
ROOT = pathlib.Path('..').resolve()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'scripts' / 'figures'))
OUTPUT = ROOT / 'output'
FIGS = ROOT / 'paper' / 'letter' / 'figures'

d = json.loads((OUTPUT / 'kd_retrieval_results.json').read_text())

from IPython.display import Image, display
import io
def show_fig(name):
    """Display a figure inline. Prefers the PNG companion; falls back to
    rendering the PDF on the fly via pymupdf if no PNG exists."""
    stem = name.replace('.pdf','').replace('.png','')
    png = FIGS / f"{stem}.png"
    if png.exists():
        display(Image(filename=str(png))); return
    pdf = FIGS / f"{stem}.pdf"
    if pdf.exists():
        try:
            import fitz
            pix = fitz.open(str(pdf))[0].get_pixmap(dpi=140)
            display(Image(data=pix.tobytes("png")))
        except Exception as e:
            print(f"  (couldn't render {pdf.name}: {e})")
    else:
        print(f"(no figure file for {stem})")


## Fig 10 — Robustness of the inter-site contrast

Panel (a) maps the K_d/Q_b degeneracy with a TwoSlopeNorm diverging
colormap. Panels (b),(c) show the joint (K_d, H) RMSE surface.

In [ ]:
from make_results_figures import fig_robustness
fig_robustness(d, FIGS / 'fig_robustness.pdf')
show_fig('fig_robustness')

## Fig 11 — alpha-sweep (Martinez per-site density retrieval)

Grants the published Martinez K(T,rho) form one per-site lever — the
deep bulk density rho_d via alpha = rho_d / 1800. Sweeps alpha against
the meter-scale-sensor RMSE at each site.

In [ ]:
import make_alpha_sweep_figure
make_alpha_sweep_figure.main()
show_fig('fig_alpha_sweep')

## Table 4 — Pooled-fit model comparison

AICc comparison of three nested models: M1 (per-site K_d, Q_b fixed),
M2 (uniform K_d, Q_b free per site), M3 (uniform K_d, Q_b fixed per
site). The DeltaAICc ~ 5 favoring M1 is what supports reading the
inter-site difference as a conductivity contrast rather than a
basal-flux contrast.

In [ ]:
import pandas as pd
# Table 4 in the paper is the POOLED three-model comparison (M1/M2/M3),
# from output/uniform_kd_test.json.
t4 = json.loads((OUTPUT / 'uniform_kd_test.json').read_text())
order = [('M1', 'M1_variable_kd',        'per-site K_d, Q_b fixed'),
         ('M2', 'M2_uniform_kd_free_qb', 'shared K_d, Q_b free'),
         ('M3', 'M3_uniform_kd_fixed_qb','shared K_d, Q_b fixed')]
rows = []
for tag, key, desc in order:
    m = t4[key]
    rows.append({'Model': f'{tag}: {desc}',
                 'RMSE (K)': round(m['rmse'], 3),
                 'Delta AICc': round(m['delta_aicc'], 2)})
pd.DataFrame(rows)

## Final sanity check — all paper figures present

In [ ]:
expected = [
    'fig_intro_probe.pdf',
    'fig_context_map.pdf',
    'fig_apollo_timeline.pdf',
    'fig_amplitude_vs_depth.pdf',
    'fig_apollo_mean_T_profile.pdf',
    'fig_kd_sweep.pdf',
    'fig_bootstrap.pdf',
    'fig_thermal_profiles.pdf',
    'fig_diviner_closure.pdf',
    'fig_robustness.pdf',
    'fig_alpha_sweep.pdf',
]
missing = [f for f in expected if not (FIGS / f).exists()]
if missing:
    print('MISSING figures:')
    for f in missing:
        print('   ', f)
else:
    print('All 11 paper figures present in paper/letter/figures/')
    print()
    print('Now compile the manuscript:')
    print('    cd paper/letter && latexmk -pdf letter.tex')